# 🚀 PARTE E: IMPLEMENTACIÓN Y SERVICIO DEL MODELO - TITANIC

## Objetivo
Crear una API REST con FastAPI para servir el modelo campeón (Regresión Logística) y realizar predicciones en tiempo real.

## Estructura de este notebook:
1. Verificar dependencias instaladas
2. Crear schema de entrada/salida
3. Implementar cliente de prueba
4. Probar la API con requests
5. Documentar resultados

In [ ]:
# ============================================
# 1. VERIFICAR DEPENDENCIAS
# ============================================

import sys
import subprocess
import pkg_resources

required = {'fastapi', 'uvicorn', 'joblib', 'pandas', 'pydantic'}
installed = {pkg.key for pkg in pkg_resources.working_set}
missing = required - installed

if missing:
    print(f" Instalando dependencias faltantes: {missing}")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
    print(" Dependencias instaladas")
else:
    print(" Todas las dependencias están instaladas")

# Verificar versiones
import fastapi
import uvicorn
import joblib
import pandas as pd
import pydantic

print(f"FastAPI: {fastapi.__version__}")
print(f"Pydantic: {pydantic.__version__}")
print(f"Joblib: {joblib.__version__}")

 Todas las dependencias están instaladas
FastAPI: 0.134.0
Pydantic: 2.12.5
Joblib: 1.5.3


In [ ]:
# ============================================
# IMPORTACIONES
# ============================================

import os
import glob
import joblib
import pandas as pd
import numpy as np
import requests
import time
import json
from datetime import datetime

print(" Librerías cargadas correctamente")
print(f" Directorio actual: {os.getcwd()}")

✅ Librerías cargadas correctamente
📂 Directorio actual: c:\Alex\Maestria\MLOPs\Final28022026\mlops-final-project-aor\notebooks


In [17]:
# ============================================
# 2. VERIFICAR MODELO Y PREPROCESADORES (CORREGIDO PARA NOTEBOOKS/)
# ============================================

import os
import glob
import joblib
import pandas as pd

print("="*60)
print(" VERIFICANDO MODELO Y PREPROCESADORES")
print("="*60)
print(f" Directorio actual: {os.getcwd()}")
print(" Nota: El notebook está en la carpeta 'notebooks/', buscando en '../models/'...\n")

# Buscar modelos en ../models/ (un nivel arriba)
model_files = glob.glob('../models/*.pkl')

if model_files:
    # Ordenar por fecha de modificación (más reciente primero)
    model_files.sort(key=os.path.getmtime, reverse=True)
    latest_model = model_files[0]
    
    print(f" Modelo encontrado: {latest_model}")
    print(f"   Archivo: {os.path.basename(latest_model)}")
    
    # Cargar modelo para verificar
    try:
        model = joblib.load(latest_model)
        print(f"\n Modelo cargado exitosamente:")
        print(f"   Tipo: {type(model).__name__}")
        
        # Verificar preprocesadores en ../data/training/ (un nivel arriba)
        print(f"\n Buscando preprocesadores en ../data/training/...")
        
        # Scaler
        scaler_path = '../data/training/scaler.pkl'
        if os.path.exists(scaler_path):
            scaler = joblib.load(scaler_path)
            print(f"    Scaler encontrado: {type(scaler).__name__}")
            print(f"      Ruta: {scaler_path}")
        else:
            print(f"    No se encontró scaler.pkl en {scaler_path}")
            # Buscar en otras ubicaciones
            alt_scaler = glob.glob('../**/scaler.pkl', recursive=True)
            if alt_scaler:
                print(f"    Scaler encontrado en: {alt_scaler[0]}")
        
        # Encoders
        encoders_path = '../data/training/encoders.pkl'
        if os.path.exists(encoders_path):
            encoders = joblib.load(encoders_path)
            print(f"    Encoders encontrados: {len(encoders)} variables")
            print(f"      Ruta: {encoders_path}")
            print(f"      Variables: {list(encoders.keys())}")
        else:
            print(f"    No se encontró encoders.pkl en {encoders_path}")
            alt_encoders = glob.glob('../**/encoders.pkl', recursive=True)
            if alt_encoders:
                print(f"    Encoders encontrados en: {alt_encoders[0]}")
        
        # Datos de entrenamiento
        X_train_path = '../data/training/X_train.csv'
        if os.path.exists(X_train_path):
            print(f"\n Datos de entrenamiento disponibles en: ../data/training/")
            X_train_sample = pd.read_csv(X_train_path).head(2)
            print(f"   Muestra de X_train (2 filas):")
            print(X_train_sample)
        else:
            print(f"\n No se encontraron datos en ../data/training/")
            
    except Exception as e:
        print(f"\n Error cargando modelo: {e}")
        
else:
    print("\n No se encontraron modelos en ../models/")
    print("\n Archivos en ../models/:")
    try:
        files = os.listdir('../models')
        for f in files:
            print(f"   - {f}")
    except:
        print("   No se puede acceder a ../models/")
    
    print("\n SOLUCIÓN:")
    print("   1. Verifica que ejecutaste: python src/train.py desde la raíz del proyecto")
    print("   2. Los modelos deben estar en: C:/Alex/Maestria/MLOPs/Final28022026/mlops-final-project-aor/models/")
    print("   3. Tu notebook está en: notebooks/, por eso usamos ../models/")

 VERIFICANDO MODELO Y PREPROCESADORES
 Directorio actual: c:\Alex\Maestria\MLOPs\Final28022026\mlops-final-project-aor\notebooks
 Nota: El notebook está en la carpeta 'notebooks/', buscando en '../models/'...

 Modelo encontrado: ../models\logistic_regression_20260228_170143.pkl
   Archivo: logistic_regression_20260228_170143.pkl

 Modelo cargado exitosamente:
   Tipo: LogisticRegression

 Buscando preprocesadores en ../data/training/...
    Scaler encontrado: StandardScaler
      Ruta: ../data/training/scaler.pkl
    Encoders encontrados: 2 variables
      Ruta: ../data/training/encoders.pkl
      Variables: ['Sex', 'Embarked']

 Datos de entrenamiento disponibles en: ../data/training/
   Muestra de X_train (2 filas):
     Pclass       Sex       Age     SibSp     Parch      Fare  Embarked  \
0  0.827377  0.737695 -0.565736  0.432793 -0.473674 -0.502445  0.585954   
1 -1.566107 -1.355574  0.663861  0.432793 -0.473674  0.786845 -1.942303   

   Has_Cabin  FamilySize   IsAlone  
0  -0.54

In [ ]:
# ============================================
# 5. VERIFICAR SERVIDOR
# ============================================
# EJECUTAR SOLO DESPUÉS DE TENER EL SERVIDOR CORRIENDO

import requests
import time

base_url = "http://localhost:8000"

print("🔍 Verificando conexión con el servidor...")

for i in range(10):
    try:
        response = requests.get(f"{base_url}/", timeout=2)
        if response.status_code == 200:
            print("\n SERVIDOR RESPONDIENDO CORRECTAMENTE")
            print(f"   Mensaje: {response.json()}")
            break
    except:
        print(f"   Intento {i+1}/10: servidor no disponible...")
        time.sleep(2)
else:
    print("\n❌ No se pudo conectar al servidor")

🔍 Verificando conexión con el servidor...

✅ SERVIDOR RESPONDIENDO CORRECTAMENTE
   Mensaje: {'message': '🚢 Titanic Survival Prediction API', 'version': '1.0.0', 'status': 'online', 'model_loaded': True, 'docs': '/docs'}


In [ ]:
# ============================================
# 6. PROBAR PREDICCIONES
# ============================================

test_cases = [
    {
        "nombre": "Jack (3ª clase, hombre joven)",
        "data": {"Pclass": 3, "Sex": "male", "Age": 20, "SibSp": 0, "Parch": 0, "Fare": 8.05, "Embarked": "S"}
    },
    {
        "nombre": "Rose (1ª clase, mujer joven)",
        "data": {"Pclass": 1, "Sex": "female", "Age": 17, "SibSp": 1, "Parch": 2, "Fare": 151.55, "Embarked": "C"}
    }
]

print(" PROBANDO PREDICCIONES")
print("="*60)

for test in test_cases:
    print(f"\n🔹 {test['nombre']}")
    try:
        response = requests.post(
            f"{base_url}/predict",
            json=test["data"],
            headers={"Content-Type": "application/json"},
            timeout=5
        )
        if response.status_code == 200:
            result = response.json()
            print(f"   → {result['survival_prediction']}")
            print(f"   → Probabilidad: {result['survival_probability']:.2%}")
        else:
            print(f"    Error: {response.status_code}")
    except Exception as e:
        print(f"    Error: {e}")

📊 PROBANDO PREDICCIONES

🔹 Jack (3ª clase, hombre joven)
   ❌ Error: 500

🔹 Rose (1ª clase, mujer joven)
   ❌ Error: 500


In [ ]:
# ============================================
# 7. GUARDAR RESULTADOS
# ============================================

import os
import json
from datetime import datetime

# Crear carpeta reports si no existe
reports_path = '../reports'
if not os.path.exists(reports_path):
    os.makedirs(reports_path)
    print(f" Carpeta creada: {reports_path}")

# Guardar resultados
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f'{reports_path}/api_test_{timestamp}.json'

# Datos a guardar
data = {
    "fecha": datetime.now().isoformat(),
    "api_url": base_url,
    "endpoints": ["/", "/health", "/predict", "/info"],
    "documentacion": {
        "swagger": f"{base_url}/docs",
        "redoc": f"{base_url}/redoc"
    }
}

# Guardar archivo
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f" Resultados guardados en: {filename}")

# Verificar que se creó
if os.path.exists(filename):
    print(f"    Tamaño: {os.path.getsize(filename)} bytes")
    print(f"    Ruta absoluta: {os.path.abspath(filename)}")

✅ Resultados guardados en: ../reports/api_test_20260228_205049.json
   📄 Tamaño: 282 bytes
   📍 Ruta absoluta: c:\Alex\Maestria\MLOPs\Final28022026\mlops-final-project-aor\reports\api_test_20260228_205049.json


In [6]:
# ============================================
# 8. DETENER SERVIDOR
# ============================================

print("""
 PARA DETENER EL SERVIDOR:

1. Ve a la terminal donde está corriendo uvicorn
2. Presiona Ctrl + C

 Servidor detenido
""")


 PARA DETENER EL SERVIDOR:

1. Ve a la terminal donde está corriendo uvicorn
2. Presiona Ctrl + C

 Servidor detenido



In [7]:
# ============================================
# 13. LIMPIEZA (OPCIONAL)
# ============================================

print("""
🧹 COMANDOS DE LIMPIEZA (ejecutar en terminal si es necesario):

# Ver puertos en uso (Windows)
netstat -ano | findstr :8000

# Matar proceso en puerto 8000 (Windows)
# Primero encuentra el PID con el comando arriba, luego:
taskkill /PID <PID> /F

# Ver logs de la API
type api.log

# Limpiar logs (si es necesario)
# del api.log
""")


🧹 COMANDOS DE LIMPIEZA (ejecutar en terminal si es necesario):

# Ver puertos en uso (Windows)
netstat -ano | findstr :8000

# Matar proceso en puerto 8000 (Windows)
# Primero encuentra el PID con el comando arriba, luego:
taskkill /PID <PID> /F

# Ver logs de la API
type api.log

# Limpiar logs (si es necesario)
# del api.log

